# Mental Health Sentiment Analysis

## Data Preprocessing and Exploratory Data Analysis (EDA)

---

### Project Overview

This notebook represents the first stage of the Mental Health Sentiment Analysis project.

Its primary objective is to transform the raw dataset into a clean, structured, and analysis-ready dataset that can be reliably used throughout the entire machine learning pipeline.

The notebook also performs a comprehensive Exploratory Data Analysis (EDA) to better understand the characteristics of the dataset, identify potential data quality issues, and uncover important patterns that may influence downstream model development.

---

### Objectives

The main objectives of this notebook are:

- Load the raw mental health dataset.
- Clean and normalize textual data.
- Remove noisy, duplicate, and invalid records.
- Standardize sentiment labels.
- Generate additional text-based features.
- Save the cleaned dataset for subsequent notebooks.
- Perform Exploratory Data Analysis (EDA).
- Visualize class distributions and text characteristics.
- Explore vocabulary patterns across different mental health categories.

---

### Expected Outputs

By the end of this notebook, the following project artifacts will be generated:

- Cleaned dataset
- Data quality report
- Statistical summaries
- Exploratory visualizations
- Word clouds
- Figures for the final report

In [1]:
# Data Manipulation
import numpy as np
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text Visualization
from wordcloud import WordCloud

# Plot Configuration
sns.set_theme(
    style="whitegrid",
    font_scale=1.1
)

plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries imported successfully.")

Libraries imported successfully.


# 2. Project Structure

To ensure consistency across all project notebooks, we define a unified project directory structure.

Using centralized paths makes the project easier to maintain, improves portability between Google Colab and Kaggle, and avoids hard-coded file locations.

In [2]:
from pathlib import Path

# Mount Google Drive (Google Colab only)

try:

    from google.colab import drive

    drive.mount("/content/drive")

    BASE_DIR = Path("/content/drive/MyDrive/Mental_Health_Classification")

except:

    print("Running outside Google Colab...")

    BASE_DIR = Path.cwd()

PROJECT_DIR = BASE_DIR

DATA_DIR = PROJECT_DIR / "data"

RAW_DATA_DIR = DATA_DIR / "raw"

PROCESSED_DATA_DIR = DATA_DIR / "processed"

MODELS_DIR = PROJECT_DIR / "models"

TOKENIZERS_DIR = PROJECT_DIR / "tokenizers"

ENCODERS_DIR = PROJECT_DIR / "encoders"

RESULTS_DIR = PROJECT_DIR / "results"

FIGURES_DIR = PROJECT_DIR / "figures"

NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"

DEPLOYMENT_DIR = PROJECT_DIR / "deployment"

# Create directories if they do not exist

directories = [

    RAW_DATA_DIR,

    PROCESSED_DATA_DIR,

    MODELS_DIR,

    TOKENIZERS_DIR,

    ENCODERS_DIR,

    RESULTS_DIR,

    FIGURES_DIR,

    NOTEBOOKS_DIR,

    DEPLOYMENT_DIR

]

for directory in directories:

    directory.mkdir(parents=True, exist_ok=True)

print("Project directories are ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directories are ready.


# 3. Load Raw Dataset

The raw dataset is loaded from the project's data directory.

Only the required columns are retained to simplify subsequent preprocessing steps.

In [3]:
DATASET_PATH = RAW_DATA_DIR / "Combined Data.csv"

df = pd.read_csv(
    DATASET_PATH
)

print(df.shape)

df.head()

(53043, 3)


,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3,I've shifted my focus to something else but I'...,Anxiety
4,4,"I'm restless and restless, it's been a month n...",Anxiety


# 4. Initial Data Inspection

Before preprocessing, we inspect the dataset structure and remove unnecessary columns that do not contribute to the sentiment classification task.

In [4]:
# Remove unnecessary index column if it exists

if "Unnamed: 0" in df.columns:

    df.drop(
        columns=["Unnamed: 0"],
        inplace=True
    )

# Create a clean sequential ID

df.insert(

    0,

    "id",

    range(len(df))

)

print("Dataset Shape :", df.shape)

print()

df.head()

Dataset Shape : (53043, 3)



,id,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3,I've shifted my focus to something else but I'...,Anxiety
4,4,"I'm restless and restless, it's been a month n...",Anxiety


# 5. Data Quality Assessment

Before cleaning the text, the dataset is inspected to ensure its overall quality.

The following checks are performed:

- Dataset dimensions
- Data types
- Missing values
- Duplicate records
- Unique sentiment classes

In [5]:
print("Dataset Shape")
print(df.shape)

print()

print("Data Types")
print(df.dtypes)

print()

print("Missing Values")
print(df.isnull().sum())

print()

print("Duplicate Rows")
print(df.duplicated().sum())

print()

print("Unique Classes")
print(sorted(df["status"].unique()))

print()

print(df["status"].value_counts())

Dataset Shape
(53043, 3)

Data Types
id            int64
statement    object
status       object
dtype: object

Missing Values
id             0
statement    362
status         0
dtype: int64

Duplicate Rows
0

Unique Classes
['Anxiety', 'Bipolar', 'Depression', 'Normal', 'Personality disorder', 'Stress', 'Suicidal']

status
Normal                  16351
Depression              15404
Suicidal                10653
Anxiety                  3888
Bipolar                  2877
Stress                   2669
Personality disorder     1201
Name: count, dtype: int64


# 6. Basic Data Cleaning

Before applying any text preprocessing techniques, the dataset is cleaned by
removing invalid records and standardizing the target labels.

This basic cleaning stage is shared across all machine learning and transformer
models.

The following operations are performed:

- Remove missing statements.
- Remove empty statements.
- Keep only the target sentiment classes.
- Reset the dataset index.
- Recreate sequential identifiers.

In [6]:
TARGET_CLASSES = [

    "Anxiety",

    "Bipolar",

    "Depression",

    "Normal",

    "Stress",

    "Suicidal"

]

# Remove missing statements

df = df.dropna(

    subset=["statement"]

)

# Remove empty statements

df["statement"] = df["statement"].astype(str).str.strip()

df = df[

    df["statement"] != ""

]

# Keep only project classes

df = df[

    df["status"].isin(TARGET_CLASSES)

].copy()

# Reset index

df.reset_index(

    drop=True,

    inplace=True

)

# Recreate IDs

df["id"] = range(len(df))

print("Dataset Shape After Cleaning")

print(df.shape)

print()

print(df["status"].value_counts())

Dataset Shape After Cleaning
(51604, 3)

status
Normal        16343
Depression    15404
Suicidal      10652
Anxiety        3841
Bipolar        2777
Stress         2587
Name: count, dtype: int64


# 7. Validate the Cleaned Dataset

After completing the basic cleaning stage, the dataset is validated once again
to ensure that no invalid records remain before text preprocessing begins.

In [7]:
print("Missing Values")

print(df.isnull().sum())

print()

print("Duplicate Rows")

print(df.duplicated().sum())

print()

print("Unique Classes")

print(sorted(df["status"].unique()))

Missing Values
id           0
statement    0
status       0
dtype: int64

Duplicate Rows
0

Unique Classes
['Anxiety', 'Bipolar', 'Depression', 'Normal', 'Stress', 'Suicidal']


# 8. Basic Text Cleaning

The preprocessing pipeline is implemented using a set of modular helper
functions.

Each function is responsible for a single text normalization task, making the
pipeline easier to understand, maintain, and reuse across different notebooks.

The complete cleaning process includes:

- HTML entity decoding
- URL removal
- Email removal
- User mention removal
- Line break normalization
- Whitespace normalization

In [8]:
import html
import re

## 8.1 Regular Expression Patterns

To improve efficiency, commonly used regular expressions are compiled once and
reused throughout the preprocessing pipeline.

In [9]:
URL_PATTERN = re.compile(

    r"http\S+|www\.\S+"
)

EMAIL_PATTERN = re.compile(

    r"\S+@\S+"
)

MENTION_PATTERN = re.compile(

    r"@\w+"
)

MULTISPACE_PATTERN = re.compile(

    r"\s+"
)

## 8.2 Text Cleaning Helper Functions

Each helper function performs one specific preprocessing task.

This modular design simplifies debugging, testing, and future extensions of
the preprocessing pipeline.

In [10]:
def decode_html(text):

    return html.unescape(text)

In [11]:
def remove_urls(text):

    return URL_PATTERN.sub("", text)

In [12]:
def remove_emails(text):

    return EMAIL_PATTERN.sub("", text)

In [13]:
def remove_mentions(text):

    return MENTION_PATTERN.sub("", text)

In [14]:
def normalize_line_breaks(text):

    return (

        text

        .replace("\n", " ")

        .replace("\r", " ")

    )

In [15]:
def normalize_spaces(text):

    return MULTISPACE_PATTERN.sub(

        " ",

        text

    ).strip()

## 8.3 Basic Cleaning Pipeline

The basic cleaning pipeline combines all helper functions into a single,
reusable preprocessing workflow.

This level of cleaning is shared across all models in the project.

In [16]:
def basic_clean_text(text):

    if pd.isna(text):

        return ""

    text = str(text)

    text = decode_html(text)

    text = remove_urls(text)

    text = remove_emails(text)

    text = remove_mentions(text)

    text = normalize_line_breaks(text)

    text = normalize_spaces(text)

    return text

# 9. Apply Basic Cleaning

The basic preprocessing pipeline is applied to every statement.

The cleaned text is stored in a new column while preserving the original raw
statement for future reference.

In [17]:
df["clean_statement"] = (

    df["statement"]

    .apply(basic_clean_text)

)

In [18]:
df = df[

    df["clean_statement"].str.len() > 1

].copy()

df.reset_index(

    drop=True,

    inplace=True

)

df["id"] = range(len(df))

In [19]:
df[
    [
        "statement",

        "clean_statement" ]

].sample(

    10,

    random_state=42
)

,statement,clean_statement
45651,Tfw you go to work sick so you can save your s...,Tfw you go to work sick so you can save your s...
41545,katortiz not forever see you soon,katortiz not forever see you soon
5488,Toy temblando kahdkakd ayuda,Toy temblando kahdkakd ayuda
44378,need a mouse look like my lappy s touch pad is...,need a mouse look like my lappy s touch pad is...
2247,"Damn it, my f is down again","Damn it, my f is down again"
20120,"I hate myself. I did not ask to be born, so I ...","I hate myself. I did not ask to be born, so I ..."
7168,"I am so fucking suicidal, but cannot bring mys...","I am so fucking suicidal, but cannot bring mys..."
41772,aaronrva is in the bathroom and i have to pee,aaronrva is in the bathroom and i have to pee
27132,"If I die, is there a way to make sure my famil...","If I die, is there a way to make sure my famil..."
38388,anytime i m alone i m instantly depressed i ca...,anytime i m alone i m instantly depressed i ca...


# 10. Machine Learning and Deep Learning Text Preprocessing

Classical machine learning models and recurrent neural networks (LSTM and
BiLSTM) benefit from a lightweight text normalization pipeline before feature
extraction or tokenization.

Unlike transformer models, these algorithms are trained on preprocessed text.

The preprocessing pipeline performs the following operations:

- Convert text to lowercase.
- Remove punctuation.
- Remove numerical values.
- Apply lemmatization.

English stopwords are intentionally preserved to retain contextual information,
which is particularly important for sentiment analysis tasks involving
negation (e.g., "not happy") and emotional expressions.

In [27]:
import string
import nltk
import string
import nltk
import re
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet")
nltk.download("omw-1.4")

LEMMATIZER = WordNetLemmatizer()

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## 10.1 Preprocessing Helper Functions

Each preprocessing step is implemented as an independent helper function to
keep the preprocessing pipeline modular, reusable, and easy to maintain.

In [24]:
# Lowercase

def to_lowercase(text):

    return text.lower()

# Remove punctuation

def remove_punctuation(text):

    return text.translate(

        str.maketrans(

            "",

            "",

            string.punctuation
        ))

# Remove numbers

def remove_numbers(text):

    return re.sub(

        r"\d+",
        "",
        text
    )

# Lemmatization

def lemmatize_text(text):

    words = [
        LEMMATIZER.lemmatize(word)

        for word in text.split()
    ]
    return " ".join(words)

## 10.2 Unified Text Preprocessing Pipeline

A single preprocessing function is provided to support multiple downstream
pipelines while avoiding duplicated code.

Two preprocessing modes are available:

- **transformer** → Basic text cleaning only (used by transformer models).
- **ml** → Additional normalization including lowercase conversion,
  punctuation removal, number removal, and lemmatization (used by classical
  machine learning models and recurrent neural networks).

This unified interface simplifies both experimentation and deployment by
providing a single entry point for text preprocessing.

In [25]:
def preprocess_text(

    text,

    mode="transformer"

):

    text = basic_clean_text(text)

    if mode == "transformer":

        return text

    if mode == "classic":

        text = to_lowercase(text)

        text = remove_punctuation(text)

        text = remove_numbers(text)

        text = lemmatize_text(text)

        return text

    raise ValueError(

        "mode must be either 'transformer' or 'classic'."

    )

## 10.3 Apply the Shared Preprocessing Pipeline

In [28]:
df["clean_statement"] = (

    df["statement"]

    .apply(

        lambda x: preprocess_text(

            x,
            mode="transformer"

        )))

df["processed_statement"] = (

    df["statement"]

    .apply(

        lambda x: preprocess_text(

            x,
            mode="classic"
        )))

In [29]:
comparison = df[
    [
        "statement",
        "clean_statement",
        "processed_statement"
    ]
].sample(

    10,
    random_state=42
)

comparison

,statement,clean_statement,processed_statement
45651,Tfw you go to work sick so you can save your s...,Tfw you go to work sick so you can save your s...,tfw you go to work sick so you can save your s...
41545,katortiz not forever see you soon,katortiz not forever see you soon,katortiz not forever see you soon
5488,Toy temblando kahdkakd ayuda,Toy temblando kahdkakd ayuda,toy temblando kahdkakd ayuda
44378,need a mouse look like my lappy s touch pad is...,need a mouse look like my lappy s touch pad is...,need a mouse look like my lappy s touch pad is...
2247,"Damn it, my f is down again","Damn it, my f is down again",damn it my f is down again
20120,"I hate myself. I did not ask to be born, so I ...","I hate myself. I did not ask to be born, so I ...",i hate myself i did not ask to be born so i sh...
7168,"I am so fucking suicidal, but cannot bring mys...","I am so fucking suicidal, but cannot bring mys...",i am so fucking suicidal but cannot bring myse...
41772,aaronrva is in the bathroom and i have to pee,aaronrva is in the bathroom and i have to pee,aaronrva is in the bathroom and i have to pee
27132,"If I die, is there a way to make sure my famil...","If I die, is there a way to make sure my famil...",if i die is there a way to make sure my family...
38388,anytime i m alone i m instantly depressed i ca...,anytime i m alone i m instantly depressed i ca...,anytime i m alone i m instantly depressed i ca...


# 11. Text Feature Engineering

In addition to the cleaned text, several numerical features are generated to
describe the length of each statement.

These features provide useful descriptive statistics during exploratory data
analysis and may also serve as additional predictors for traditional machine
learning models.

The following features are computed:

- Character count
- Word count

In [30]:
df["char_len"] = (

    df["clean_statement"]

    .str.len()
)

df["word_len"] = (

    df["clean_statement"]

    .str.split()

    .str.len()
)

In [31]:
df[
    [
        "clean_statement",

        "processed_statement",

        "char_len",

        "word_len"
    ]
].head()

,clean_statement,processed_statement,char_len,word_len
0,oh my gosh,oh my gosh,10,3
1,"trouble sleeping, confused mind, restless hear...",trouble sleeping confused mind restless heart ...,64,10
2,"All wrong, back off dear, forward doubt. Stay ...",all wrong back off dear forward doubt stay in ...,78,14
3,I've shifted my focus to something else but I'...,ive shifted my focus to something else but im ...,61,11
4,"I'm restless and restless, it's been a month n...",im restless and restless it been a month now b...,72,14


# 12. Organize the Final Dataset

The processed dataset is reorganized into a consistent column order before
being saved for downstream modeling.

In [32]:
df = df[
    [
        "id",

        "statement",

        "clean_statement",

        "processed_statement",

        "status",

        "char_len",

        "word_len"
    ]]

In [33]:
df.head()

,id,statement,clean_statement,processed_statement,status,char_len,word_len
0,0,oh my gosh,oh my gosh,oh my gosh,Anxiety,10,3
1,1,"trouble sleeping, confused mind, restless hear...","trouble sleeping, confused mind, restless hear...",trouble sleeping confused mind restless heart ...,Anxiety,64,10
2,2,"All wrong, back off dear, forward doubt. Stay ...","All wrong, back off dear, forward doubt. Stay ...",all wrong back off dear forward doubt stay in ...,Anxiety,78,14
3,3,I've shifted my focus to something else but I'...,I've shifted my focus to something else but I'...,ive shifted my focus to something else but im ...,Anxiety,61,11
4,4,"I'm restless and restless, it's been a month n...","I'm restless and restless, it's been a month n...",im restless and restless it been a month now b...,Anxiety,72,14


# 13. Save Project Assets

The processed dataset is saved as the official input for all subsequent
notebooks.

Rather than repeating the preprocessing pipeline multiple times, every machine
learning, deep learning, and transformer notebook will directly load this
dataset.

This ensures consistency across experiments and avoids unnecessary computation.

In [34]:
PREPROCESSED_DATA_PATH = PROCESSED_DATA_DIR / "preprocessed_data.csv"

df.to_csv(

    PREPROCESSED_DATA_PATH,

    index=False,

    encoding="utf-8-sig"

)

print(f"Dataset saved to:\n{PREPROCESSED_DATA_PATH}")

Dataset saved to:
/content/drive/MyDrive/Mental_Health_Classification/data/processed/preprocessed_data.csv


In [35]:
# Verify the Saved Dataset

verification_df = pd.read_csv(

    PREPROCESSED_DATA_PATH
)
verification_df.head()

print(verification_df.shape)
print()
print(verification_df.columns.tolist())

(51599, 7)

['id', 'statement', 'clean_statement', 'processed_statement', 'status', 'char_len', 'word_len']


## Dataset Summary

The following summary provides a quick overview of the processed dataset that
will be used throughout the remainder of the project.

In [41]:
print("=" * 60)

print("Processed Dataset Summary")

print("=" * 60)

print(f"Samples : {len(df):,}")

print(f"Classes : {df['status'].nunique()}")

print()

print(df["status"].value_counts())

print()

print(df.columns.tolist())

Processed Dataset Summary
Samples : 51,599
Classes : 6

status
Normal        16339
Depression    15404
Suicidal      10651
Anxiety        3841
Bipolar        2777
Stress         2587
Name: count, dtype: int64

['id', 'statement', 'clean_statement', 'processed_statement', 'status', 'char_len', 'word_len']


# 14. Export Preprocessing Utilities

To eliminate duplicated preprocessing code across multiple notebooks and the
deployment application, the text preprocessing functions are stored in a
separate Python module.

This module becomes the single source of truth for text preprocessing and will
later be imported by:

- Machine Learning notebooks
- Deep Learning notebooks
- Transformer notebooks
- Model comparison notebook
- FastAPI deployment

Maintaining a shared preprocessing module improves code reusability,
maintainability, and consistency throughout the project.

In [36]:
UTILS_DIR = PROJECT_DIR / "utils"

UTILS_DIR.mkdir(

    parents=True,

    exist_ok=True

)

print("Utilities directory is ready.")

Utilities directory is ready.


In [37]:
INIT_FILE = UTILS_DIR / "__init__.py"

INIT_FILE.touch(exist_ok=True)

print("__init__.py created successfully.")

__init__.py created successfully.


In [38]:
import sys

if str(PROJECT_DIR) not in sys.path:

    sys.path.append(str(PROJECT_DIR))

from utils.preprocessing import preprocess_text

In [40]:
sample = "I'm NOT happy!!! Visit https://google.com 123"

print(

    preprocess_text(

        sample,

        mode="transformer"

    )
)

print()

print(

    preprocess_text(

        sample,
        mode="classic"
    )
)

I'm NOT happy!!! Visit 123

im not happy visit
